

## Ejercicio 4

Resuelva el punto c) del ejercicio anterior con las imágenes originales de las carpetas **train** y **test** y utilice.

### a)

el objeto **ImageDataGenerator** del módulo **tensorflow.keras.preprocessing.image** para generar de forma automática una versión aumentada de los datos con las características del punto b). Utilice el método **flow_from_file** del objeto **ImageDataGenerator** para utilizar directamente las imágenes de las carpetas en vez de cargarlas en memoria.


In [1]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import Sequential, optimizers, callbacks
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, LeakyReLU

# --- Parámetros generales ---
EPOCAS = 100
LOTES  = 128
PACIENCIA = 5
ACTIVA = LeakyReLU()
IMG_SIZE = (128, 128)  # ajustá según tamaño de tus imágenes del dataset Fingers
N_CLASSES = 6          # si tus clases son 0, 1, 2, 3, 4, 5 (por cantidad de dedos)

# --- Directorios ---
DATASET_DIR = '../imagenes/Fingers/'  # ajustá según tu estructura
TRAIN_DIR = DATASET_DIR + 'train/'
TEST_DIR  = DATASET_DIR + 'test/'

# === Generadores de datos ===

# Generador con aumento de datos (rotaciones, etc.)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,      # rotaciones aleatorias entre -45° y +45°
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='reflect',
    validation_split=0.2     # usa una parte del train como validación
)

# Generador para test (sin aumento)
test_datagen = ImageDataGenerator(rescale=1./255)

# Crea generadores a partir de carpetas
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    color_mode='grayscale',   # o 'rgb' según corresponda
    batch_size=LOTES,
    class_mode='sparse',
    subset='training',
    shuffle=True
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=LOTES,
    class_mode='sparse',
    subset='validation',
    shuffle=True
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=LOTES,
    class_mode='sparse',
    shuffle=False
)

# === Define el modelo ===
model_gen = Sequential()
model_gen.add(Input(shape=IMG_SIZE + (1,)))  # + (1,) porque es grayscale
model_gen.add(Conv2D(16, kernel_size=(3,3), strides=(1,1), padding='same', activation=ACTIVA))
model_gen.add(MaxPooling2D(pool_size=(2,2)))
model_gen.add(Conv2D(32, kernel_size=(3,3), strides=(1,1), padding='same', activation=ACTIVA))
model_gen.add(MaxPooling2D(pool_size=(2,2)))
model_gen.add(Flatten())
model_gen.add(Dense(32, activation=ACTIVA))
model_gen.add(Dense(N_CLASSES, activation='softmax'))

model_gen.summary()

# === Compila el modelo ===
optimizer = optimizers.Adam(0.001)
model_gen.compile(optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# === Early stopping ===
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=PACIENCIA,
    min_delta=0.001,
    restore_best_weights=True
)

# === Entrena el modelo ===
H_gen = model_gen.fit(
    train_generator,
    epochs=EPOCAS,
    validation_data=val_generator,
    callbacks=[early_stop],
    verbose=1
)

# === Evalúa con el test set ===
pred_gen = model_gen.evaluate(test_generator, verbose=0)
print("Efectividad del modelo entrenado con ImageDataGenerator: %6.2f%%" % (pred_gen[1]*100))


2025-10-18 13:44:08.740060: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-18 13:44:08.740277: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-18 13:44:08.773712: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-18 13:44:09.667523: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

Found 14400 images belonging to 6 classes.
Found 3600 images belonging to 6 classes.
Found 3600 images belonging to 6 classes.


2025-10-18 13:44:10.843236: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 128, 16)   │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │     1,048,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

 Non-trainable params: 0 (0.00 B)

/home/manuel/Documents/Facultad/DeepLearning/env/lib/python3.13/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/100


E0000 00:00:1760805851.862592  100870 meta_optimizer.cc:967] remapper failed: INVALID_ARGUMENT: Mutation::Apply error: fanout 'StatefulPartitionedCall/gradient_tape/sequential_1/conv2d_1_2/leaky_re_lu_1/LeakyRelu/LeakyReluGrad' exist for missing node 'StatefulPartitionedCall/sequential_1/conv2d_1_2/BiasAdd'.


113/113 ━━━━━━━━━━━━━━━━━━━━ 46s 400ms/step - accuracy: 0.4633 - loss: 1.3345 - val_accuracy: 0.6739 - val_loss: 0.8522
Epoch 2/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 49s 434ms/step - accuracy: 0.7611 - loss: 0.6532 - val_accuracy: 0.8497 - val_loss: 0.4784
Epoch 3/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 47s 416ms/step - accuracy: 0.8585 - loss: 0.4240 - val_accuracy: 0.8919 - val_loss: 0.3333
Epoch 4/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 45s 397ms/step - accuracy: 0.8913 - loss: 0.3274 - val_accuracy: 0.9167 - val_loss: 0.2486
Epoch 5/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 44s 390ms/step - accuracy: 0.9181 - loss: 0.2452 - val_accuracy: 0.9067 - val_loss: 0.2595
Epoch 6/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 44s 390ms/step - accuracy: 0.9320 - loss: 0.2083 - val_accuracy: 0.9531 - val_loss: 0.1624
Epoch 7/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 43s 384ms/step - accuracy: 0.9476 - loss: 0.1664 - val_accuracy: 0.9494 - val_loss: 0.1512
Epoch 8/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 44s 390ms/step - accuracy: 0.9565 - loss: 0.136

### Guardado del modelo

In [2]:
model_gen.save('Modelos/Fingers_4_conv_model.keras')

### b)

Repita el punto a) utilizando la función **tf.keras.utils.image_dataset_from_directory** para generar el dataset a partir de carpetas en combinación con funciones para aplicar el preprocesamiento/data augmentation.

In [3]:
# ====== EJERCICIO 4.b ======

import tensorflow as tf
from tensorflow.keras import Sequential, optimizers, callbacks
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Flatten, Dense, LeakyReLU,
    RandomFlip, RandomRotation, RandomZoom, RandomTranslation
)

# --- Parámetros generales ---
EPOCAS = 100
LOTES  = 128
PACIENCIA = 5
ACTIVA = LeakyReLU()
IMG_SIZE = (128, 128)
N_CLASSES = 6

# --- Directorios ---
DATASET_DIR = '../imagenes/Fingers/'
TRAIN_DIR = DATASET_DIR + 'train/'
TEST_DIR  = DATASET_DIR + 'test/'

# === Carga de datasets desde carpetas ===
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    color_mode='grayscale',  # o 'rgb' según corresponda
    batch_size=LOTES,
    label_mode='int',
    validation_split=0.2,
    subset='training',
    seed=123
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=LOTES,
    label_mode='int',
    validation_split=0.2,
    subset='validation',
    seed=123
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=LOTES,
    label_mode='int'
)

# Normalización automática de valores a [0,1]
normalization_layer = tf.keras.layers.Rescaling(1./255)

# === Data augmentation ===
data_augmentation = Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.25),        # ~ ±45°
    RandomZoom(0.2),
    RandomTranslation(0.1, 0.1),
])

# === Aplica normalización y data augmentation ===
train_dataset = train_dataset.map(lambda x, y: (normalization_layer(data_augmentation(x)), y))
val_dataset   = val_dataset.map(lambda x, y: (normalization_layer(x), y))
test_dataset  = test_dataset.map(lambda x, y: (normalization_layer(x), y))

# Mejora rendimiento (prefetch)
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset   = val_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset  = test_dataset.prefetch(buffer_size=AUTOTUNE)

# === Modelo CNN (igual que antes) ===
model_ds = Sequential()
model_ds.add(Input(shape=IMG_SIZE + (1,)))
model_ds.add(Conv2D(16, kernel_size=(3,3), strides=(1,1), padding='same', activation=ACTIVA))
model_ds.add(MaxPooling2D(pool_size=(2,2)))
model_ds.add(Conv2D(32, kernel_size=(3,3), strides=(1,1), padding='same', activation=ACTIVA))
model_ds.add(MaxPooling2D(pool_size=(2,2)))
model_ds.add(Flatten())
model_ds.add(Dense(32, activation=ACTIVA))
model_ds.add(Dense(N_CLASSES, activation='softmax'))

model_ds.summary()

# === Compila el modelo ===
optimizer = optimizers.Adam(0.001)
model_ds.compile(optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# === Early stopping ===
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=PACIENCIA,
    min_delta=0.001,
    restore_best_weights=True
)

# === Entrena el modelo ===
H_ds = model_ds.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCAS,
    callbacks=[early_stop],
    verbose=1
)

# === Evalúa con el test set ===
pred_ds = model_ds.evaluate(test_dataset, verbose=0)
print("Efectividad del modelo con image_dataset_from_directory: %6.2f%%" % (pred_ds[1]*100))


Found 18000 files belonging to 6 classes.
Using 14400 files for training.
Found 18000 files belonging to 6 classes.
Using 3600 files for validation.
Found 3600 files belonging to 6 classes.


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 128, 128, 16)   │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 64, 64, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │     1,048,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,053,606 (4.02 MB)

 Trainable params: 1,053,606 (4.02 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100


E0000 00:00:1760808753.598726  100870 meta_optimizer.cc:967] remapper failed: INVALID_ARGUMENT: Mutation::Apply error: fanout 'StatefulPartitionedCall/gradient_tape/sequential_2_1/conv2d_3_1/leaky_re_lu_1_1/LeakyRelu/LeakyReluGrad' exist for missing node 'StatefulPartitionedCall/sequential_2_1/conv2d_3_1/BiasAdd'.


113/113 ━━━━━━━━━━━━━━━━━━━━ 36s 316ms/step - accuracy: 0.3346 - loss: 1.5689 - val_accuracy: 0.4639 - val_loss: 1.0713
Epoch 2/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 41s 359ms/step - accuracy: 0.6084 - loss: 0.9634 - val_accuracy: 0.6586 - val_loss: 0.8316
Epoch 3/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 40s 352ms/step - accuracy: 0.7353 - loss: 0.6852 - val_accuracy: 0.7606 - val_loss: 0.5844
Epoch 4/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 39s 340ms/step - accuracy: 0.7988 - loss: 0.5359 - val_accuracy: 0.7881 - val_loss: 0.4919
Epoch 5/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 44s 385ms/step - accuracy: 0.8478 - loss: 0.4278 - val_accuracy: 0.9547 - val_loss: 0.2405
Epoch 6/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 38s 335ms/step - accuracy: 0.8678 - loss: 0.3714 - val_accuracy: 0.8131 - val_loss: 0.4338
Epoch 7/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 38s 335ms/step - accuracy: 0.8978 - loss: 0.3041 - val_accuracy: 0.8408 - val_loss: 0.3626
Epoch 8/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 39s 345ms/step - accuracy: 0.9024 - loss: 0.278

### Guardado del modelo

In [4]:
model_ds.save('Modelos/Fingers_4.b_conv_model.keras')